# GuindiMa AI — échantillons de voix wolof (voie XTTS, sur Colab)

`tests/generer_echantillons_voix.py` tourne en local sur le PC (SpeechT5 + x-vecteurs) :
~600 Mo, pas de souci avec 8 Go de RAM. Ce notebook-ci prépare la **seule**
voix qu'on ne peut pas produire confortablement en local : **XTTS-v2-wolof**
(`muhamadul/xtts-v2-wolof`), ~467 M paramètres, **1,7 Go de poids**, 24 kHz.
Chargé sur un PC à 8 Go avec l'OS et le reste, c'est trop juste sur CPU ; sur
Colab (GPU gratuit), c'est immédiat.

XTTS sait **cloner une voix** : on lui donne un court extrait d'une voix
féminine jeune, et il lit la phrase de réponse avec ce timbre. C'est la piste
la plus sûre pour une voix plus jeune que les deux x-vecteurs CMU ARCTIC.

Même phrase que les échantillons locaux (ligne 46) :

> `Jëlal bus bu Tata AFTU, ligne ñeent fukk ak juróom benn.`

**Licence** : modèle XTTS-v2 sous *Coqui Public Model License* (CPML),
**non commercial**. À garder pour un usage de démonstration / recherche.

In [ ]:
# Vérifier qu'un GPU est bien attribué (sinon Runtime > Change runtime type > GPU).
!nvidia-smi

In [ ]:
# coqui-tts est la version maintenue de TTS (le paquet officiel TTS n'est plus publié).
!pip install -q coqui-tts

In [ ]:
# Récupérer le modèle fine-tuné pour le wolof.
from huggingface_hub import snapshot_download

snapshot_download("muhamadul/xtts-v2-wolof", local_dir="xtts-v2-wolof")

import os
for nom in sorted(os.listdir("xtts-v2-wolof")):
    print(nom)

In [ ]:
# Charger XTTS avec les poids fine-tunés (pas besoin de retélécharger la base).
import torch
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

config = XttsConfig()
config.load_json("xtts-v2-wolof/config.json")
config.model_args.output_sample_rate = 24000

model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir="xtts-v2-wolof/", eval=True)
model.cuda()

print("XTTS-v2-wolof chargé")

## Voix de référence à cloner

Dépose un extrait **de 6 à 15 secondes** d'une voix féminine jeune, si possible
en wolof (sans musique, sans bruit). XTTS en copie le timbre.

À défaut de référence wolof, une voix féminine jeune d'une autre langue marche
aussi — le modèle a été fine-tuné avec LJSpeech (voix féminine jeune, anglais),
il sait donc reprendre ce type de timbre.

In [ ]:
import os
from google.colab import files

if not os.path.exists("reference.wav"):
    print("Choisis un fichier audio de référence (6 à 15 s, voix féminine jeune)…")
    envoi = files.upload()
    nom = next(iter(envoi))
    os.rename(nom, "reference.wav")

print("Référence :", os.path.abspath("reference.wav"))

In [ ]:
import soundfile as sf

PHRASE = "Jëlal bus bu Tata AFTU, ligne ñeent fukk ak juróom benn."

# XTTS-v2 n'a pas de jeton de langue pour le wolof : on passe le français,
# ce que recommande la fiche du modèle (proximité phonologique).
sortie = model.synthesize(
    PHRASE,
    config,
    speaker_wav="reference.wav",
    language="fr",
)
sf.write("04_xtts_feminine_jeune.wav", sortie["wav"], 24000)
print("Écrit : 04_xtts_feminine_jeune.wav")

In [ ]:
from IPython.display import Audio, display
from google.colab import files

display(Audio("04_xtts_feminine_jeune.wav"))
files.download("04_xtts_feminine_jeune.wav")